# 基于CANN快速跑通第一个Ascend C算子
本快速入门指南主要介绍基于 CANN 的 Ascend C 算子开发基础流程，核心内容及步骤如下：

1. 环境准备：配置程序运行所需的环境变量

2. 算子分析：梳理目标算子的设计规格

3. 核函数开发：实现算子核心功能代码

4. 算子调用：开发算子调用逻辑代码

5. 编译运行：编译算子与调用代码并执行，完成功能验证

本样例的算子实现（Device 侧）与调用代码（Host 侧）**支持混合编译在同一源文件**（`Sources/add.asc`）中，并通过内核调用符 [`<<<...>>>`](https://asc.gitcode.com/guide/programming_guide/language_extension/simd_builtin_keywords.html) 调用核函数，可快速完成算子的开发与功能验证。本次快速入门的所有开发工作，均基于 `Sources/add.asc` 文件开展。

---

## 1. 环境准备
正式开始快速入门之前，先要对jupyter环境进行初始化。以下代码完成了初始化并将环境中的变量导入jupyter环境，同时完成了代码目录的创建。保证能正常导入代码以及使用bisheng编译器，完成算子的开发及编译。

In [ ]:
!mkdir -p Sources

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("\n🎉 Environment initialization process completed successfully!")

---

## 2. 算子分析
快速入门场景，基于Add算子为例进行自定义算子开发全流程介绍，为简化学习流程，该自定义算子存在以下特点。

- 输入[shape](https://asc.gitcode.com/guide/technical_appendix/concepts_and_terms/neural_networks_and_operators/operator_basics.html)固定为（8，2048）

- 输入类型仅支持float

- 固定使用8个核

基于上述条件，算子设计规格如下所示：

<table style="float: left; border-collapse: collapse; margin: 0 10px 10px 0; font-size: 14px;">
<tr style="background: #f0f0f0;">
    <td align="center"><a href="https://asc.gitcode.com/guide/technical_appendix/concepts_and_terms/neural_networks_and_operators/operator_basics.html">算子类型(OpType)</a></td>
    <td colspan="4" align="center">Add</td>
</tr>
<tr>
    <td rowspan="3" align="center">算子输入</td>
    <td align="center">name</td>
    <td align="center">shape</td>
    <td align="center">data type</td>
    <td align="center"><a href="https://asc.gitcode.com/guide/technical_appendix/concepts_and_terms/neural_networks_and_operators/data_layout.html">format</a></td>
</tr>
<tr>
    <td align="center">x</td>
    <td align="center">(8, 2048)</td>
    <td align="center">float</td>
    <td align="center">ND</td>
</tr>
<tr>
    <td align="center">y</td>
    <td align="center">(8, 2048)</td>
    <td align="center">float</td>
    <td align="center">ND</td>
</tr>
<tr>
    <td rowspan="1" align="center">算子输出</td>
    <td align="center">z</td>
    <td align="center">(8, 2048)</td>
    <td align="center">float</td>
    <td align="center">ND</td>
</tr>
<tr>
    <td rowspan="1" align="center">核函数名</td>
    <td colspan="4" align="center">add</td>
</tr>
<tr>
    <td rowspan="1" align="center">使用核数</td>
    <td colspan="4" align="center">8</td>
</tr>
</table>


---

## 3. 核函数开发
本样例使用固定 8 个核并行计算：数据整体长度为 8 * 2048 个元素，平均分配到 8 个核上，每个核处理 2048 个元素。Ascend C 核函数是一个核上的处理函数，只处理本核分到的数据。


### 3.1 头文件引入
进行算子开发时，首先要在add.asc源文件中导入必要的头文件。这些头文件都是固定头文件，在进行其它自定义算子开发时可直接复用。  

In [ ]:
%%writefile Sources/add.asc

#include <cstdint>
#include <iostream>
#include <vector>
#include <algorithm>
#include <iterator>
#include "acl/acl.h"
#include "kernel_operator.h"

### 3.2 核函数实现
[核函数（Kernel）](https://asc.gitcode.com/guide/programming_guide/programming_model/ai_core_simd_programming/kernel_function.html) 是 Ascend C 算子设备侧实现的入口，由主机端代码调用，调用时被实例化为多份并行副本，分别调度到多个 AI Core 上运行（每个 AI Core 被抽象为一个 [Block](https://asc.gitcode.com/guide/technical_appendix/concepts_and_terms/glossary.html)）。

定义核函数需遵循以下规则：

- 必须使用 [`__global__`](https://asc.gitcode.com/guide/programming_guide/language_extension/simd_builtin_keywords.html) 函数类型限定符标识它是核函数，可以被 `<<<...>>>` 调用；
- 还需根据算子实际执行的硬件单元选择一种函数执行空间限定符：[`__aicore__`](https://asc.gitcode.com/guide/programming_guide/language_extension/simd_builtin_keywords.html)（不区分核类型）、[`__vector__`](https://asc.gitcode.com/guide/programming_guide/language_extension/simd_builtin_keywords.html)（Vector 核）、[`__cube__`](https://asc.gitcode.com/guide/programming_guide/language_extension/simd_builtin_keywords.html)（Cube 核）、[`__mix__`](https://asc.gitcode.com/guide/programming_guide/language_extension/simd_builtin_keywords.html)（Cube 与 Vector 融合）。本算子仅包含 Vector 计算，使用 `__vector__`；
- 指针入参需要增加变量类型限定符 [`__gm__`](https://asc.gitcode.com/guide/programming_guide/language_extension/simd_builtin_keywords.html)，表明该指针变量指向 Global Memory；
- 核函数必须具有 void 返回类型，参数列表需遵循相应限制。

核函数的计算逻辑遵循「搬入-计算-搬出」三段式流水结构。

前置说明：

- **[GM（Global Memory）](https://asc.gitcode.com/guide/technical_appendix/concepts_and_terms/glossary.html)**：AI Core 外部的全局存储，通过 [`AscendC::GlobalTensor`](https://asc.gitcode.com/api/SIMD-API/basic_api/data_structures/GlobalTensor/GlobalTensor_intro.html) 访问，容量大、访问速度较慢；
- **[UB（Unified Buffer）](https://asc.gitcode.com/guide/technical_appendix/concepts_and_terms/glossary.html)**：AI Core 内部的向量计算专用缓存，通过 [`AscendC::LocalTensor`](https://asc.gitcode.com/api/SIMD-API/basic_api/data_structures/LocalTensor/LocalTensor_intro.html) 访问，容量有限、访问速度快；
- **[DataCopy](https://asc.gitcode.com/api/SIMD-API/basic_api/memory_vector_compute/data_move/DataCopy_GMAndUB_continuous.html)**：用于在 GM 和 UB 之间搬运数据的接口，搬运方向由参数顺序决定；
- **[PipeBarrier](https://asc.gitcode.com/api/SIMD-API/basic_api/sync_control/intra_core_sync/PipeBarrier_ISASI.html)**：流水线同步屏障，确保数据搬运完成后再执行后续操作，避免读写冲突；
- **[block_idx](https://asc.gitcode.com/guide/programming_guide/language_extension/simd_builtin_keywords.html)**：当前核的编号，用于多核并行时的数据分片计算。

具体实现：

- **初始化**：通过 [`AscendC::InitSocState()`](https://asc.gitcode.com/api/SIMD-API/basic_api/tool_interface/system_init/InitSocState.html) 初始化 AI Core 硬件状态；
- **数据分块（Tiling）**：按 `block_idx * blockLength` 偏移，通过 `AscendC::GlobalTensor` 的 [`SetGlobalBuffer`](https://asc.gitcode.com/api/SIMD-API/basic_api/data_structures/GlobalTensor/SetGlobalBuffer.html) 设定本核在 GM 上的数据段：

<img src="./images/inter_kernel_partition.png" alt="inter_kernel_partition"  width="700px" >

- **片上内存分配**：通过 [`AscendC::LocalMemAllocator<AscendC::Hardware::UB>`](https://asc.gitcode.com/api/SIMD-API/basic_api/resource_management/LocalMemAllocator/LocalMemAllocator_intro.html) 在 UB 上为 x、y、z 各分配一块 `LocalTensor`；
- **搬入**：通过 `AscendC::DataCopy` 将输入 x、y 从 GM 搬运到 UB；
- **计算**：通过 [`AscendC::Add`](https://asc.gitcode.com/api/SIMD-API/basic_api/memory_vector_compute/basic_arithmetic/Add.html) 在 UB 上完成 z = x + y；
- **搬出**：通过 `AscendC::DataCopy` 将结果 z 从 UB 搬运回 GM；
- 各阶段之间使用 `AscendC::PipeBarrier<PIPE_ALL>()` 同步。

核函数实现如下：

> **说明**
> - 本样例中搬入、计算、搬出三个阶段严格串行执行，各硬件单元（MTE2 / Vector / MTE3）无法同时工作；实际开发中可采用 Ping-Pong 双缓冲机制，使三个阶段并行执行，隐藏搬运延迟。
> - Tiling（数据切分）参数通常由 Host 侧计算后传递给核函数（Kernel）；本样例输入 shape 固定、切分策略固定，Host 侧 Tiling 实现可省略，直接通过模板参数 `blockLength` 指定每个核处理的数据长度。


In [ ]:
%%writefile -a Sources/add.asc

// 核函数入口：使用 __vector__ 声明为矢量核，模板参数 blockLength 为每个核处理的数据长度
template <uint32_t blockLength>
__vector__ __global__ void add_custom(__gm__ float *x, __gm__ float *y, __gm__ float *z)
{
    AscendC::InitSocState();  // 初始化 SoC 状态

    // 设定本核在 Global Memory 上的数据起始地址（按核号 block_idx 偏移）及长度
    AscendC::GlobalTensor<float> xGm, yGm, zGm;
    xGm.SetGlobalBuffer(x + block_idx * blockLength, blockLength);
    yGm.SetGlobalBuffer(y + block_idx * blockLength, blockLength);
    zGm.SetGlobalBuffer(z + block_idx * blockLength, blockLength);

    // 在 UB 上按数组方式分配输入/输出 LocalTensor
    AscendC::LocalMemAllocator<AscendC::Hardware::UB> ubAllocator;
    AscendC::LocalTensor<float> xLocal = ubAllocator.Alloc<float, blockLength>();
    AscendC::LocalTensor<float> yLocal = ubAllocator.Alloc<float, blockLength>();
    AscendC::LocalTensor<float> zLocal = ubAllocator.Alloc<float, blockLength>();

    // 搬入：Global Memory -> Local Memory，并用 PipeBarrier 同步
    AscendC::DataCopy(xLocal, xGm, blockLength);
    AscendC::DataCopy(yLocal, yGm, blockLength);
    AscendC::PipeBarrier<PIPE_ALL>();

    // 计算：逐元素相加，并用 PipeBarrier 同步
    AscendC::Add(zLocal, xLocal, yLocal, blockLength);
    AscendC::PipeBarrier<PIPE_ALL>();

    // 搬出：Local Memory -> Global Memory，并用 PipeBarrier 同步
    AscendC::DataCopy(zGm, zLocal, blockLength);
    AscendC::PipeBarrier<PIPE_ALL>();
}


---

## 4. 算子调用
完成Kernel侧核函数开发后，即可编写Host侧的核函数调用程序。实现从Host侧的APP程序调用算子，执行计算过程。本算子中主要分为以下三个部分。

1. 核函数调用：通过<<<...>>>内核调用符进行算子调用

2. 计算结果比对：比对golden数据和实际输出，验证算子精度

3. 算子验证主程序：生成输入及golden数据，用来进行算子验证

### 4.1 核函数调用
Host 端通过内核调用符 `<<<...>>>` 的语法形式调用核函数（Kernel），语法如下：

```
kernel_name<<<numBlocks, dynUBufSize, stream>>>(argument list);
```

`<<<...>>>` 内的执行配置由 3 个参数决定：

- **[numBlocks](https://asc.gitcode.com/guide/programming_guide/language_extension/simd_builtin_keywords.html)**：规定了核函数将会在几个核上执行；每个执行该核函数的核会被分配一个逻辑 ID，即 block_idx，可以在核函数中使用内置变量 `block_idx` 获取。本样例固定为 8；
- **[dynUBufSize](https://asc.gitcode.com/guide/programming_guide/language_extension/simd_builtin_keywords.html)**：Dynamic Unified Buffer（UB）Size，配置 UB 动态内存分配空间的大小（bytes），本样例采用静态分配，设置为 0；
- **[stream](https://asc.gitcode.com/guide/programming_guide/language_extension/simd_builtin_keywords.html)**：类型为 aclrtStream，用于维护异步操作的执行顺序，确保按照应用程序中的代码调用顺序在 device 上执行。

本样例调用示例如下：

```
// 模板参数 blockLength=2048 为每个核处理的数据长度
add_custom<blockLength><<<numBlocks, 0, stream>>>(xDevice, yDevice, zDevice);
```

核函数的调用是异步的，调用结束后控制权立刻返回主机端，可以调用以下 aclrtSynchronizeStream 函数来强制主机端程序等待所有核函数执行完毕。

```
aclError aclrtSynchronizeStream(aclrtStream stream);
```

在整个调用过程中，需要申请 Host 和 Device 内存，在 Host 侧读入数据后，将数据拷贝到 Device 侧，进而调用 `<<<...>>>` 进行算子执行，其逻辑流程图如下所示：

<img src="./images/kernel_function_call_code_logic_block_diagram.png" alt="kernel_function_call_code_logic_block_diagram"  width="200px" >

本样例实现如下：


In [ ]:
%%writefile -a Sources/add.asc

std::vector<float> kernel_add(std::vector<float> &x, std::vector<float> &y)
{
    constexpr uint32_t numBlocks = 8;
    constexpr uint32_t blockLength = 2048;
    uint32_t totalLength = x.size();
    size_t totalByteSize = totalLength * sizeof(float);
    int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    uint8_t *zHost = nullptr;
    float *xDevice = nullptr;
    float *yDevice = nullptr;
    float *zDevice = nullptr;

    // 初始化
    aclInit(nullptr);
    // 运行管理资源申请
    aclrtSetDevice(deviceId);
    aclrtCreateStream(&stream);
    // 分配Host内存
    aclrtMallocHost((void **)(&zHost), totalByteSize);
    // 分配Device内存
    aclrtMalloc((void **)&xDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void **)&yDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void **)&zDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    // 将Host上的输入数据拷贝到Device侧
    aclrtMemcpy(xDevice, totalByteSize, x.data(), totalByteSize, ACL_MEMCPY_HOST_TO_DEVICE);
    aclrtMemcpy(yDevice, totalByteSize, y.data(), totalByteSize, ACL_MEMCPY_HOST_TO_DEVICE);
    // 用内核调用符<<<...>>>调用核函数完成指定的运算
    add_custom<blockLength><<<numBlocks, 0, stream>>>(xDevice, yDevice, zDevice);
    aclrtSynchronizeStream(stream);
    // 将Device上的运算结果拷贝回Host
    aclrtMemcpy(zHost, totalByteSize, zDevice, totalByteSize, ACL_MEMCPY_DEVICE_TO_HOST);
    std::vector<float> z((float *)zHost, (float *)zHost + totalLength);
    // 释放申请的资源
    aclrtFree(xDevice);
    aclrtFree(yDevice);
    aclrtFree(zDevice);
    aclrtFreeHost(zHost);
    // 去初始化
    aclrtDestroyStream(stream);
    aclrtResetDevice(deviceId);
    aclFinalize();
    return z;
}


在实现其他算子时，这里大部分代码均可复用，需要修改的点如下：

1. 需要根据实际输入及输出个数，调整 Host 和 Device 的内存申请、拷贝、释放以及 `<<<...>>>` 中的参数。
2. 需要根据实际要使用的核数，定义 `numBlocks`。
3. 需要根据实际数据长度，调整数据分块（Tiling）：定义模板参数 `blockLength` 与动态 UB 大小。


### 4.2 计算结果比对
传入实际输出与golden值，进行一一比对，若不相同，则证明存在精度问题。本样例中会同时将output和golden中前20个数打印出来进行查看。

In [ ]:
%%writefile -a Sources/add.asc

uint32_t VerifyResult(std::vector<float> &output, std::vector<float> &golden)
{
    auto printTensor = [](std::vector<float> &tensor, const char *name) {
        constexpr size_t maxPrintSize = 20;
        std::cout << name << ": ";
        std::copy(tensor.begin(), tensor.begin() + std::min(tensor.size(), maxPrintSize),
            std::ostream_iterator<float>(std::cout, " "));
        if (tensor.size() > maxPrintSize) {
            std::cout << "...";
        }
        std::cout << std::endl;
    };
    printTensor(output, "Output");
    printTensor(golden, "Golden");
    if (output.size() == golden.size() && std::equal(output.begin(), output.end(), golden.begin())) {
        std::cout << "[Success] Case accuracy is verification passed." << std::endl;
        return 0;
    } else {
        std::cout << "[Failed] Case accuracy is verification failed!" << std::endl;
        return 1;
    }
    return 0;
}

### 4.3 算子验证主程序
定义输入数据长度，并生成实际的golden数据。

In [ ]:
%%writefile -a Sources/add.asc

int32_t main(int32_t argc, char *argv[])
{
    constexpr uint32_t totalLength = 8 * 2048;
    constexpr float valueX = 1.2f;
    constexpr float valueY = 2.3f;
    std::vector<float> x(totalLength, valueX);
    std::vector<float> y(totalLength, valueY);

    std::vector<float> output = kernel_add(x, y);

    std::vector<float> golden(totalLength, valueX + valueY);
    return VerifyResult(output, golden);
}

---

## 5. 编译运行
完成所有代码的开发工作后，即可进行编译运行。首先执行以下代码编译可执行文件：

In [ ]:
!bisheng Sources/add.asc --npu-arch=dav-2201 -o add_custom

再执行以下代码，进行算子的实际运行。

In [ ]:
!./add_custom